# Case Study 01 — Feature Engineering: WOE/IV & Scorecard Preparation

Transforms raw features into model-ready inputs using:
- Weight of Evidence (WOE) encoding
- Information Value (IV) for feature selection
- Monotonic binning for interpretable scorecard

**Prerequisite:** Run `01_eda.ipynb` first and ensure data is loaded.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

REPO_ROOT = Path().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from utils.preprocessing import woe_binning, iv_summary

# ── Load data (same as EDA notebook) ──
DATA_DIR = Path('../data')
candidates = list(DATA_DIR.glob('*.xls')) + list(DATA_DIR.glob('*.xlsx')) + list(DATA_DIR.glob('*.csv'))
fpath = candidates[0]
raw = pd.read_excel(fpath, header=1) if fpath.suffix in ('.xls', '.xlsx') else pd.read_csv(fpath)
df = raw.copy()
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('.', '_')
target_raw = [c for c in df.columns if 'default' in c][0]
df = df.rename(columns={target_raw: 'default', 'id': 'client_id'})
TARGET = 'default'
FEATURES = [c for c in df.columns if c not in [TARGET, 'client_id']]
print(f'Data loaded: {df.shape}')

## 1. Information Value — Feature Selection Screen

In [ ]:
iv_df = iv_summary(df[FEATURES + [TARGET]], TARGET, bins=10)

# Visualise IV
TEAL = '#01696f'; MAROON = '#a12c7b'; ORANGE = '#964219'; GRAY = '#bab9b4'
color_map = {'Strong': MAROON, 'Medium': TEAL, 'Weak': ORANGE, 'Useless': GRAY}

fig, ax = plt.subplots(figsize=(10, 7), facecolor='white')
bars = ax.barh(iv_df['feature'], iv_df['iv'],
              color=[color_map.get(s, GRAY) for s in iv_df['strength']],
              edgecolor='none')
ax.axvline(0.02, color=GRAY, lw=1, linestyle='--', label='Weak threshold (0.02)')
ax.axvline(0.10, color=ORANGE, lw=1, linestyle='--', label='Medium threshold (0.10)')
ax.axvline(0.30, color=MAROON, lw=1, linestyle='--', label='Strong threshold (0.30)')
ax.set_xlabel('Information Value (IV)')
ax.set_title('Feature Selection — Information Value Screen')
ax.legend(fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
ax.spines[['left', 'bottom']].set_color('#dcd9d5')
ax.set_facecolor('white')
plt.tight_layout()
plt.savefig('../reports/07_iv_feature_selection.png', bbox_inches='tight', dpi=150)
plt.show()

print(iv_df.to_string(index=False))

## 2. WOE Encoding — Selected Features

In [ ]:
# Select features with IV > 0.02 (at least Weak predictive power)
selected = iv_df[iv_df['iv'] > 0.02]['feature'].tolist()
print(f'Selected features (IV > 0.02): {len(selected)}')
print(selected)

# WOE-encode selected features
df_woe = df[selected + [TARGET]].copy()
woe_maps = {}

for feat in selected:
    woe_df = woe_binning(df, feat, TARGET, bins=10)
    # Build a mapping: bin interval → WOE value
    woe_maps[feat] = woe_df.set_index('bin')['woe'].to_dict()
    df_woe[f'{feat}_woe'] = pd.cut(
        df[feat], bins=pd.IntervalIndex(woe_df['bin'].values), include_lowest=True
    ).map(woe_maps[feat]).fillna(0)

woe_cols = [c for c in df_woe.columns if c.endswith('_woe')]
print(f'\nWOE-encoded features: {len(woe_cols)}')
df_woe[woe_cols + [TARGET]].head()

In [ ]:
# Save processed dataset for modeling notebook
OUT_PATH = DATA_DIR / 'processed_woe.parquet'
df_woe[woe_cols + [TARGET]].to_parquet(OUT_PATH, index=False)
print(f'Processed dataset saved → {OUT_PATH}')
print(f'Shape: {df_woe[woe_cols + [TARGET]].shape}')